# House Price Prediction using Machine Learning

# 1. Import Required Libraries

In this section, we import all libraries required for:

- Data Manipulation
- Visualization
- Machine Learning
- Model Evaluation
- Deployment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Load Dataset

The Ames Housing Dataset contains information about residential properties.

Dataset Statistics:

- Total Rows: 1460
- Total Features: 81
- Target Variable: SalePrice

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# 3. Dataset Exploration

We first inspect:

- Dataset Shape
- Data Types
- Statistical Summary
- Duplicate Records

In [ ]:
train.head()

In [ ]:
train.shape

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.duplicated().sum()

In [ ]:
train.drop("Id", axis=1, inplace=True)
test.drop("Id", axis=1, inplace=True)

# 4. Missing Value Analysis

Missing values are common in real-world datasets.

Before building machine learning models, we must identify:

- Which columns contain missing values
- Percentage of missing data
- Whether the missing values represent actual absence or incomplete information

In [ ]:
X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]

IDENTIFY NUMERICAL AND CATEGORICAL FEATURES

In [ ]:
num_cols = X.select_dtypes(include=['int64','float64']).columns
print(num_cols)

In [ ]:
cat_cols = X.select_dtypes(include=['object']).columns
print(cat_cols)

In [ ]:
print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

- Percentage of missing data
 - Whether the missing values represent actual absence or incomplete information

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0]
missing.sort_values(ascending=False)

In [ ]:
missing_percent = (
    train.isnull().sum() /
    len(train)
)*100

missing_percent = missing_percent[
    missing_percent > 0
].sort_values(
    ascending=False
)

print(missing_percent)

## Handling Missing Values for Features Representing Absence

Some categorical features contain missing values that do not actually indicate missing information. Instead, they represent the absence of a particular property feature.

Examples include:

- No Basement
- No Garage
- No Masonry Veneer
- No Fireplace
- No Pool
- No Fence
- No Alley Access
- No Miscellaneous Feature

For such features, missing values are replaced with **"None"** so that the model can correctly interpret the absence of that feature rather than treating it as unknown data.

In [ ]:
bsmt_cols = [
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2'
]

for col in bsmt_cols:
    train[col] = train[col].fillna("None")
    test[col] = test[col].fillna("None")

garage_cat_cols = [
    'GarageType',
    'GarageFinish',
    'GarageQual',
    'GarageCond'
]

for col in garage_cat_cols:
    train[col] = train[col].fillna("None")
    test[col] = test[col].fillna("None")

train["MasVnrType"] = train["MasVnrType"].fillna("None")
test["MasVnrType"] = test["MasVnrType"].fillna("None")

train["FireplaceQu"] = train["FireplaceQu"].fillna("None")
test["FireplaceQu"] = test["FireplaceQu"].fillna("None")

train["PoolQC"] = train["PoolQC"].fillna("None")
test["PoolQC"] = test["PoolQC"].fillna("None")

train["Fence"] = train["Fence"].fillna("None")
test["Fence"] = test["Fence"].fillna("None")

train["Alley"] = train["Alley"].fillna("None")
test["Alley"] = test["Alley"].fillna("None")

train["MiscFeature"] = train["MiscFeature"].fillna("None")
test["MiscFeature"] = test["MiscFeature"].fillna("None")

### Fill Numerical Features Where Absence Means Zero

Some numerical features contain missing values because the corresponding property feature does not exist.

Examples:

- No Garage → GarageCars = 0, GarageArea = 0
- No Garage Built Year → GarageYrBlt = 0
- No Masonry Veneer → MasVnrArea = 0

Since the absence of these features naturally corresponds to a value of zero, missing values are replaced with **0** instead of using mean or median imputation.

In [ ]:
garage_num_cols = [
    'GarageYrBlt',
    'GarageCars',
    'GarageArea'
]

for col in garage_num_cols:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

train["MasVnrArea"] = train["MasVnrArea"].fillna(0)
test["MasVnrArea"] = test["MasVnrArea"].fillna(0)

bsmt_num_cols = [
    'BsmtFinSF1',
    'BsmtFinSF2',
    'BsmtUnfSF',
    'TotalBsmtSF',
    'BsmtFullBath',
    'BsmtHalfBath'
]

for col in bsmt_num_cols:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

### Statistical Imputation for Remaining Missing Values

After handling features where missing values represented the absence of a property characteristic, a small number of missing values still remained.

Different statistical techniques were applied based on the data type:

- **LotFrontage** was filled using the **median** value because it is a numerical feature and median is less sensitive to outliers.
- Remaining **categorical features** were filled using the **mode** (most frequent value) to preserve the existing distribution of categories.

This approach ensures that all remaining missing values are handled without introducing significant bias into the dataset.

In [ ]:
train["LotFrontage"] = train["LotFrontage"].fillna(
    train["LotFrontage"].median()
)

test["LotFrontage"] = test["LotFrontage"].fillna(
    test["LotFrontage"].median()
)

for col in train.select_dtypes(include='object'):
    train[col] = train[col].fillna(
        train[col].mode()[0]
    )

for col in test.select_dtypes(include='object'):
    test[col] = test[col].fillna(
        test[col].mode()[0]
    )

### Final Numerical Feature Imputation

After applying feature-specific imputation strategies, a few numerical features may still contain missing values.

To ensure a complete dataset, all remaining numerical features are filled using their **median values**.

Why Median?

- Less affected by extreme values and outliers.
- Preserves the central tendency of the data.
- More robust than mean for skewed distributions.

This step guarantees that no numerical missing values remain before model training.

In [ ]:
for col in train.select_dtypes(
    include=['int64', 'float64']
):
    train[col] = train[col].fillna(
        train[col].median()
    )

for col in test.select_dtypes(
    include=['int64', 'float64']
):
    test[col] = test[col].fillna(
        test[col].median()
    )

Verify That No Missing Values Remain

In [ ]:
train.isnull().sum().sum()
test.isnull().sum().sum()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(train["SalePrice"], kde=True)
plt.show()

## Log Transformation of Target Variable

The distribution of **SalePrice** is positively skewed, meaning that a small number of houses have extremely high prices.

Machine learning models often perform better when the target variable follows a more normal distribution.

To reduce skewness and stabilize variance, a logarithmic transformation is applied using:

\[
y = \log(1 + SalePrice)
\]

This transformation helps improve model performance and makes predictions more reliable.

In [ ]:
y = np.log1p(train["SalePrice"])

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(y, kde=True)
plt.show()

### Features Most Correlated with SalePrice

The following analysis ranks numerical features based on their correlation with the target variable.

Features with higher positive correlation generally have a stronger impact on house prices and are likely to be important predictors for the machine learning models.

Correlation with SalePrice


In [ ]:
corr_matrix = train.corr(numeric_only=True)

saleprice_corr = (
    corr_matrix["SalePrice"]
    .sort_values(ascending=False)
)

print(saleprice_corr)

Top 10 Most Important Correlations

In [ ]:
plt.figure(figsize=(10,6))

saleprice_corr[1:11].plot(
    kind='bar'
)

plt.show()

## Relationship Between Important Features and SalePrice

To better understand the factors influencing house prices, we visualize the relationship between some of the most important features and the target variable (**SalePrice**).

The selected features include:

- **GrLivArea** (Above Ground Living Area)
- **OverallQual** (Overall Quality Rating)
- **GarageCars** (Garage Capacity)
- **YearBuilt** (Construction Year)

These visualizations help identify:

- Feature impact on house prices
- Trends and patterns
- Potential outliers
- Feature importance for model building

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(40,10))

sns.scatterplot(
    x=train["GrLivArea"],
    y=train["SalePrice"],
    ax=axes[0]
)
axes[0].set_title("GrLivArea vs SalePrice")

sns.boxplot(
    x=train["OverallQual"],
    y=train["SalePrice"],
    ax=axes[1]
)
axes[1].set_title("OverallQual vs SalePrice")

sns.boxplot(
    x=train["GarageCars"],
    y=train["SalePrice"],
    ax=axes[2]
)
axes[2].set_title("GarageCars vs SalePrice")

sns.scatterplot(
    x=train["YearBuilt"],
    y=train["SalePrice"],
    ax=axes[3]
)
axes[3].set_title("YearBuilt vs SalePrice")

plt.tight_layout()
plt.show()

### Key Observations

- Houses with larger **GrLivArea** generally have higher sale prices.
- Higher **OverallQual** ratings are strongly associated with increased house prices.
- Properties with larger garage capacity tend to be more expensive.
- Newer houses (**YearBuilt**) generally have higher market values.
- A few extreme observations (outliers) are visible and may require further investigation.

## Outlier Detection and Removal

During exploratory data analysis, a few properties were identified with unusually large living areas but relatively low sale prices.

These observations do not follow the general market trend and may negatively affect model performance.

To improve model robustness, such extreme outliers are removed from the training dataset.

In [ ]:
train = train.drop(
    train[
        (train["GrLivArea"] > 4000) &
        (train["SalePrice"] < 300000)
    ].index
)

### Feature Distribution Analysis

After handling missing values and removing extreme outliers, we visualize the distribution of numerical features.

This helps us understand:

- Data spread and variability
- Presence of skewed distributions
- Potential remaining outliers
- Overall feature characteristics

In [ ]:
train.hist(
    figsize=(20,20),
    bins=30
)

plt.show()

Relationship Between Important Features and Price

In [ ]:


# Most Important Features
# OverallQual
# GrLivArea
# GarageCars
# GarageArea
# TotalBsmtSF
# 1stFlrSF
# FullBath
# YearBuilt
# Outliers Exist


FEATURE ENGINEERING

In [ ]:
#Combine Train and Test Data
train_len = len(train)

combined = pd.concat(
    [train.drop("SalePrice", axis=1),
     test],
    axis=0
).reset_index(drop=True)

## Convert Numerical Categories to Categorical Features

Some features are stored as numerical values but actually represent categories rather than quantities.

Examples:

- **MSSubClass** → Type of dwelling
- **MoSold** → Month Sold
- **YrSold** → Year Sold

Although these features contain numbers, the numerical order does not carry meaningful magnitude information.

Therefore, they are converted to **string (categorical) data types** so that they can be properly encoded during feature engineering.

In [ ]:
combined["MSSubClass"] = (
    combined["MSSubClass"]
    .astype(str)
)
combined["MoSold"] = (
    combined["MoSold"]
    .astype(str)
)
combined["YrSold"] = (
    combined["YrSold"]
    .astype(str)
)

## Creating New Features

Feature engineering involves creating new meaningful features from existing data to improve model performance.

Instead of relying only on the original dataset attributes, additional features are generated to better capture property characteristics that influence house prices.

The following engineered features are created:

### Area-Based Features
- **TotalSF** : Total square footage of the house.
- **TotalBath** : Total effective bathrooms in the property.
- **TotalPorchSF** : Combined area of all porch and deck spaces.

### Age-Based Features
- **HouseAge** : Age of the house at the time of sale.
- **RemodelAge** : Number of years since the last remodeling.
- **GarageAge** : Age of the garage at the time of sale.

### Binary Features
- **HasBsmt** : Indicates whether the property has a basement.
- **HasGarage** : Indicates whether the property has a garage.
- **HasFireplace** : Indicates whether the property has a fireplace.
- **HasPool** : Indicates whether the property has a swimming pool.

These engineered features provide additional information that may not be directly captured by the original features and can improve predictive performance.

In [ ]:
combined["TotalSF"] = (
    combined["TotalBsmtSF"]
    + combined["1stFlrSF"]
    + combined["2ndFlrSF"]
)
combined["TotalBath"] = (
    combined["FullBath"]
    + 0.5 * combined["HalfBath"]
    + combined["BsmtFullBath"]
    + 0.5 * combined["BsmtHalfBath"]
)
combined["TotalPorchSF"] = (
    combined["OpenPorchSF"]
    + combined["3SsnPorch"]
    + combined["EnclosedPorch"]
    + combined["ScreenPorch"]
    + combined["WoodDeckSF"]
)
combined["HouseAge"] = (
    combined["YrSold"].astype(int)
    - combined["YearBuilt"]
)
combined["RemodelAge"] = (
    combined["YrSold"].astype(int)
    - combined["YearRemodAdd"]
)
combined["GarageAge"] = (
    combined["YrSold"].astype(int)
    - combined["GarageYrBlt"]
)
combined["HasBsmt"] = (
    combined["TotalBsmtSF"] > 0
).astype(int)
combined["HasGarage"] = (
    combined["GarageArea"] > 0
).astype(int)
combined["HasFireplace"] = (
    combined["Fireplaces"] > 0
).astype(int)
combined["HasPool"] = (
    combined["PoolArea"] > 0
).astype(int)

## Encoding Categorical Features

Machine learning models require numerical input, but the dataset contains several categorical features represented as text values.

To make these features usable for model training, **One-Hot Encoding** is applied.

### Steps Performed

1. Combine training and testing datasets to ensure consistent encoding.
2. Identify all categorical features.
3. Apply One-Hot Encoding using `pd.get_dummies()`.
4. Remove one dummy variable from each category (`drop_first=True`) to avoid multicollinearity.

After encoding, all categorical variables are converted into numerical features that can be processed by machine learning algorithms.

In [ ]:
train_len = len(train)

combined = pd.concat(
    [train.drop("SalePrice", axis=1), test],
    axis=0
).reset_index(drop=True)

cat_cols = combined.select_dtypes(
    include='object'
).columns

print(cat_cols)
combined = pd.get_dummies(
    combined,
    drop_first=True
)

combined.shape

Split Back into Train and Test

In [ ]:
X = combined.iloc[:train_len, :]
X_test = combined.iloc[train_len:, :]

In [ ]:
X.head()

## Train-Validation Split and Feature Scaling

Before training machine learning models, the dataset is divided into training and validation sets.

### Train-Validation Split

The dataset is split into:

- **80% Training Data**
- **20% Validation Data**

The training set is used to train the models, while the validation set is used to evaluate their performance on unseen data.

### Feature Scaling

Some machine learning algorithms, such as **Linear Regression**, perform better when features are on a similar scale.

To standardize the feature values, **StandardScaler** is applied, which transforms the data to have:

- Mean = 0
- Standard Deviation = 1

This helps improve model convergence and ensures that features with larger magnitudes do not dominate the learning process.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(X_train.shape)
print(X_val.shape)

print(X_train_scaled.shape)
print(X_val_scaled.shape)

Linear Regression



In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Train model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Prediction
y_pred_lr = lr_model.predict(X_val_scaled)

# Evaluation
mae_lr = mean_absolute_error(y_val, y_pred_lr)
mse_lr = mean_squared_error(y_val, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_val, y_pred_lr)

print("Linear Regression Results")
print("MAE :", mae_lr)
print("MSE :", mse_lr)
print("RMSE:", rmse_lr)
print("R² Score:", r2_lr)

Random forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Train model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

# Prediction
y_pred_rf = rf_model.predict(X_val)

# Evaluation
mae_rf = mean_absolute_error(y_val, y_pred_rf)
mse_rf = mean_squared_error(y_val, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_val, y_pred_rf)

print("Random Forest Results")
print("MAE :", mae_rf)
print("MSE :", mse_rf)
print("RMSE:", rmse_rf)
print("R² Score:", r2_rf)

XGBoost Regressor

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Train model
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

# Prediction
y_pred_xgb = xgb_model.predict(X_val)

# Evaluation
mae_xgb = mean_absolute_error(y_val, y_pred_xgb)
mse_xgb = mean_squared_error(y_val, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
r2_xgb = r2_score(y_val, y_pred_xgb)

print("XGBoost Results")
print("MAE :", mae_xgb)
print("MSE :", mse_xgb)
print("RMSE:", rmse_xgb)
print("R² Score:", r2_xgb)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],

    "MAE": [
        mae_lr,
        mae_rf,
        mae_xgb
    ],

    "MSE": [
        mse_lr,
        mse_rf,
        mse_xgb
    ],

    "RMSE": [
        rmse_lr,
        rmse_rf,
        rmse_xgb
    ],

    "R2 Score": [
        r2_lr,
        r2_rf,
        r2_xgb
    ]
})

comparison

In [ ]:
import matplotlib.pyplot as plt

models = [
    "Linear Regression",
    "Random Forest",
    "XGBoost"
]

r2_scores = [
    r2_lr,
    r2_rf,
    r2_xgb
]

plt.figure(figsize=(8,5))

plt.bar(models, r2_scores)

plt.xlabel("Models")
plt.ylabel("R² Score")
plt.title("Comparison of Models")

plt.ylim(0,1)

plt.show()

 input   and saving the most accurate model which is XGBOOST 
 

In [ ]:
new_house = X.iloc[[0]].copy()

In [ ]:
new_house["OverallQual"] = 7
new_house["GrLivArea"] = 1800
new_house["GarageCars"] = 2
new_house["YearBuilt"] = 2005
new_house['1stFlrSF']=2000

In [ ]:
prediction = xgb_model.predict(new_house)

predicted_price = np.expm1(prediction)

print("Predicted House Price:", predicted_price[0])

In [ ]:
import pickle

# Save models
pickle.dump(xgb_model, open("xgb_model.pkl", "wb"))
pickle.dump(rf_model, open("rf_model.pkl", "wb"))
pickle.dump(lr_model, open("lr_model.pkl", "wb"))

# Save scaler (used for Linear Regression)
pickle.dump(scaler, open("scaler.pkl", "wb"))

In [ ]:
import pickle

model = pickle.load(open("xgb_model.pkl", "rb"))

print(model)

In [ ]:
import pickle

# Model training ke final dataset se ek sample row
template_row = X.iloc[0].copy()

pickle.dump(template_row, open("template_row.pkl", "wb"))

print("template_row.pkl saved successfully")

In [ ]:
model = pickle.load(open("xgb_model.pkl", "rb"))

In [ ]:
import pandas as pd
import pickle

default_row = pd.Series(0, index=feature_names)

pickle.dump(default_row, open("default_row.pkl", "wb"))

print("Saved")

In [ ]:
import pandas as pd
import pickle

default_row = X.median(
    numeric_only=True
)

for col in X.columns:
    if col not in default_row.index:
        default_row[col] = 0

default_row = default_row.reindex(
    X.columns
)

pickle.dump(
    default_row,
    open("default_row.pkl","wb")
)

print("default_row.pkl saved")

In [ ]:
default_row = X.median(numeric_only=True)

for col in X.columns:
    if col not in default_row.index:
        default_row[col] = 0

default_row = default_row.reindex(X.columns)

pickle.dump(default_row, open("default_row.pkl", "wb"))

In [ ]:
import pandas as pd
import pickle

# Model ke saare features
features = model.feature_names_in_

# Har feature ko default 0 do
default_row = pd.Series(0, index=features)

# Save kar do
pickle.dump(default_row, open("default_row.pkl", "wb"))

print("Saved Successfully")
print("Total Features:", len(default_row))

In [ ]:
import pickle

row = pickle.load(open("default_row.pkl", "rb"))

print(row.head())
print(len(row))